## Fitting auditory population activity

This tutorial will show you how to fit a model on the NS1 dataset in population coding mode.

Let's start with some boilerplate imports.

In [5]:
import os
import wandb
import torch.utils

from deepSTRF.datasets.audio import NS1_DRC_Dataset_pop
from deepSTRF.models.audio import ConvNet2D
from deepSTRF.utils.training_pop import optimize_multiple_seeds

Let's check whether a GPU is available for PyTorch. 

In [2]:
device = torch.device('cuda:0') if torch.cuda.is_available() else torch.device('cpu')
print(f"\nSelected device: {device}\n")


Selected device: cpu



Let's now instanciate the dataset. For this, we need to define the path to the data (cf. NS1 dataset setup instructions).

In [9]:
# dataset
root = '/home/ulysse/PycharmProjects/deepSTRF/deepSTRF/'
neuron_indices = tuple(range(73))
dataset = NS1_DRC_Dataset_pop(path=root + 'datasets/audio/NS1_DRC/data', stimuli=('nat',), neuron_indexes=neuron_indices, normalize_resps=False)
F = dataset.get_F()     # number of spectrogram frequency bands
N = dataset.get_N()     # total number of available neurons to fit in the dataset

FileNotFoundError: [Errno 2] No such file or directory: '/home/ulysse/PycharmProjects/deepSTRF/deepSTRF/datasets/audio/NS1_DRC/data/ns1_drc_spectrograms.pt'

We also need to define a function to split the data into train, validation and test set, given our instanciated datasets.

In [ ]:
def data_split_func(datasets):
    est_set = datasets[0]
    train_set, valid_set = torch.utils.data.random_split(est_set, [int(0.8 * len(est_set)), len(est_set) - int(0.8 * len(est_set))])
    test_set = datasets[1]
    return train_set, valid_set, test_set

Some training hyperparameters; cross-validation across 3 seeds.

In [ ]:
# optimization
seeds = list(range(3))
n_epochs_early_stop = 50
batch_size = 16
learning_rate = 0.001
weight_decay = 0.
criterion = torch.nn.MSELoss()

Now we can instanciate our model. Similar to above, we need to wrap this into a function which will be used at the beginning of the training process in different random seeds.

In [11]:
# prefiltering / parameterization (if applies) / model architecture hyperparameters
T = 15  # Temporal window size
prefilt_dict = {'type': 'AdapTrans', 'dt': 10.0, 'min_freq': 500, 'max_freq': 20000, 'scale': 'mel'}
param_dict = {'type': 'DCLS', 'num_gauss': 10}
def model_init_fn():
    """ change it as you like """
    # model = Linear(n_frequency_bands=F, temporal_window_size=T, out_neurons=N, prefiltering=prefilt_dict, parameterization=param_dict)
    # model = LinearNonlinear(n_frequency_bands=49, temporal_window_size=T, prefiltering=prefilt_dict, parameterization=param_dict)
    # model = NetworkReceptiveField(n_frequency_bands=49, temporal_window_size=T, n_hidden=20, prefiltering=prefilt_dict, parameterization=param_dict)
    # model = DNet(n_frequency_bands=F, temporal_window_size=T, n_hidden=20, init_tau=2., prefiltering=prefilt_dict, parameterization=param_dict)
    model = ConvNet2D(n_frequency_bands=F, kernel_size=(3, 7), c_hidden=10, n_hidden=90, out_neurons=N, prefiltering=None)
    model = model.to(device)
    return model
net = model_init_fn()
print(net)

NameError: name 'F' is not defined

Log some hyperparameter values about the run on wandb; feel free to adapt it to your liking.

In [ ]:
# Weights & Biases logging
os.environ["WANDB_MODE"] = "offline"  # comment out for online logging
project_name = 'deepSTRF'
entity_name = 'urancon'
config = {
    "seeds": seeds,
    "temporal_window_size": T,
    "learning_rate": learning_rate,
    "weight_decay": weight_decay,
    "model": net.__class__.__name__,
    "Nb of parameters": net.count_trainable_params(),
    "Dataset": est_set.__class__.__name__
}
wandb.init(project=project_name, entity=entity_name, config=config)
print(f"Model: {net.__class__.__name__}, # params: {net.count_trainable_params()}")

In [ ]:
# create folder for model saves
savedir = os.path.join(root, f'results/{est_set.__class__.__name__}/{net.__class__.__name__}/')
if not os.path.exists(savedir):
    os.makedirs(savedir)

We are finally ready to launch the training process. We first need to define which neurons (i.e., our population) we want to fit our model on.

In [ ]:
print("\n#### POPULATION ####\n")
print("Neuron indices:\n", neuron_indices)
est_set.select_population(neuron_indices)
val_set.select_population(neuron_indices)# train model on multiple splits of this population's data and average results

In [ ]:
pop_res_dict = optimize_multiple_seeds(0, seeds, datasets,
                                       data_split_func, model_init_fn,
                                       criterion, learning_rate, weight_decay, batch_size, n_epochs_early_stop, device, savedir
                                       )